In [1]:
pip install transformers torch textstat nltk scipy


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 4.9 MB/s  0:00:005.2 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [textstat]
Note: you may need to restart the kernel to use updated packages.


In [2]:
import nltk
nltk.download("vader_lexicon")


[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /home/deepak/nltk_data...


True

In [3]:
import textstat

def readability_score(text: str) -> float:
    """
    Returns Flesch Reading Ease score.
    Higher = easier to read
    """
    return textstat.flesch_reading_ease(text)


In [4]:
from nltk.sentiment import SentimentIntensityAnalyzer

sia = SentimentIntensityAnalyzer()

def sentiment_score(text: str) -> float:
    """
    Returns compound sentiment score in range [-1, 1]
    """
    return sia.polarity_scores(text)["compound"]


In [5]:
from transformers import pipeline

toxicity_pipeline = pipeline(
    "text-classification",
    model="unitary/toxic-bert",
    tokenizer="unitary/toxic-bert",
    return_all_scores=True
)

def toxicity_score(text: str) -> float:
    """
    Returns probability of toxicity [0,1]
    """
    scores = toxicity_pipeline(text)[0]
    for s in scores:
        if s["label"].lower() == "toxic":
            return s["score"]
    return 0.0


2026-01-17 14:27:40.263161: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-17 14:27:40.294486: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-01-17 14:27:41.106309: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


config.json:   0%|          | 0.00/811 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/174 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Device set to use cpu
/home/deepak/anaconda3/lib/python3.13/site-packages/transformers/pipelines/text_classification.py:111: UserWarning: `return_all_scores` is now deprecated,  if want a similar functionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores=False`.
  warnings.warn(


In [6]:
import torch
from transformers import GPT2LMHeadModel, GPT2TokenizerFast
from scipy.special import expit

tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")
model.eval()

def detected_synthetic_score(text: str) -> float:
    """
    Returns probability [0,1] that text is synthetic (AI-generated)
    """
    encodings = tokenizer(text, return_tensors="pt")
    with torch.no_grad():
        outputs = model(**encodings, labels=encodings["input_ids"])
        loss = outputs.loss.item()

    perplexity = torch.exp(torch.tensor(loss)).item()

    # Map perplexity → probability
    # Low perplexity = more synthetic
    score = expit((50 - perplexity) / 10)
    return float(score)


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [15]:
from datetime import datetime, timezone
import math
import dateutil.parser

def news_freshness_score(published_at: str, tau_hours: int = 72) -> float:
    """
    Computes recency score in [0,1].
    Higher = more recent news.
    """
    if not published_at:
        return 0.0

    try:
        published_time = dateutil.parser.parse(published_at)
        if published_time.tzinfo is None:
            published_time = published_time.replace(tzinfo=timezone.utc)

        now = datetime.now(timezone.utc)
        age_hours = (now - published_time).total_seconds() / 3600

        return math.exp(-age_hours / tau_hours)
    except Exception:
        return 0.0


In [12]:
def extract_text_features(text: str) -> dict:
    return {
        "readability_score": readability_score(text),
        "sentiment_score": sentiment_score(text),
        "toxicity_score": toxicity_score(text),
        "detected_synthetic_score": detected_synthetic_score(text),
        "text_length": len(text)
    }


In [13]:
import requests

SERPER_URL = "https://google.serper.dev/news"

def fetch_news_text(topic, num_results=5):
    headers = {
        "X-API-KEY": "3cd9effd78d88fd5ac0b20a280f2a58e2c2a10c6",
        "Content-Type": "application/json"
    }

    payload = {
        "q": topic,
        "num": num_results
    }

    response = requests.post(SERPER_URL, headers=headers, json=payload)
    response.raise_for_status()
    data = response.json()

    results = []
    for item in data.get("news", []):
        results.append({
            "title": item.get("title", ""),
            "snippet": item.get("snippet", ""),
            "url": item.get("link", "")
        })

    return results


In [9]:
def fetch_news_with_features(topic, num_results=5):
    news_items = fetch_news_text(topic, num_results)
    enriched_results = []

    for item in news_items:
        # Combine title + snippet for analysis
        text = f"{item['title']}. {item['snippet']}"

        features = extract_text_features(text)

        enriched_results.append({
            "title": item["title"],
            "snippet": item["snippet"],
            "url": item["url"],
            **features
        })

    return enriched_results


In [14]:
results = fetch_news_with_features("us attacks Iran again", num_results=5)

for r in results:
    print(r)


{'title': 'Trump Threatens New Military Action Against Iran', 'snippet': "January/February 2026. By Kelsey Davenport. President Donald Trump said the United States will support new Israeli strikes against Iran's...", 'url': 'https://www.armscontrol.org/act/2026-01/news/trump-threatens-new-military-action-against-iran', 'readability_score': 41.853717948717986, 'sentiment_score': 0.1027, 'toxicity_score': 0.0019199107773602009, 'detected_synthetic_score': 0.4937479416873309, 'text_length': 190, 'news_freshness_score': 0.0}
{'title': 'For Now, the US Holds Off on Attacking Iran', 'snippet': 'For now, President Donald Trump says he will “wait and see” before striking Iran, adding that “sources” have told him that the Iranian...', 'url': 'https://www.stimson.org/2026/for-now-the-us-holds-off-on-attacking-iran/', 'readability_score': 68.9825, 'sentiment_score': -0.4588, 'toxicity_score': 0.010222058743238449, 'detected_synthetic_score': 0.22532284823970955, 'text_length': 182, 'news_freshnes

In [31]:
import ollama
import json

def analyze_news_llama3(news_item: dict, user_claim: str) -> dict:
    """
    Claim-aware misinformation analysis.
    """

    prompt = f"""
You are a professional fact-checker.

USER CLAIM:
"{user_claim}"

NEWS ARTICLE:
Title: {news_item['title']}
Snippet: {news_item['snippet']}

TASK:
Determine how the article relates to the USER CLAIM.

Step-by-step:
1. Identify the main claim in the USER CLAIM.
2. Determine whether the article:
   - SUPPORTS the claim
   - REFUTES the claim
   - DOES NOT ADDRESS the claim
3. Based on that relationship, decide if the USER CLAIM is REAL or FAKE.

IMPORTANT:
- The article being factual does NOT mean the claim is real.
- Debunking articles imply the claim is FAKE.
- If unclear, return UNKNOWN.

AUXILIARY SIGNALS (supporting only):
- Readability: {news_item['readability_score']}
- Sentiment: {news_item['sentiment_score']}
- Toxicity: {news_item['toxicity_score']}
- Synthetic probability: {news_item['detected_synthetic_score']}

OUTPUT JSON ONLY:
{{
  "relationship": "SUPPORTS" | "REFUTES" | "NOT_ADDRESSED",
  "verdict": "REAL" | "FAKE" | "UNKNOWN",
  "confidence": 0 to 1,
  "reasoning": "concise explanation"
}}
"""

    response = ollama.chat(
        model="qwen2.5:14b",
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": 0.1}
    )

    try:
        return json.loads(response["message"]["content"])
    except Exception:
        return {
            "relationship": "UNKNOWN",
            "verdict": "UNKNOWN",
            "confidence": 0.0,
            "reasoning": "Invalid model output"
        }


In [32]:
def aggregate_claim_verdict(analyses: list) -> dict:
    supports = sum(a["relationship"] == "SUPPORTS" for a in analyses)
    refutes = sum(a["relationship"] == "REFUTES" for a in analyses)

    if refutes > supports:
        return {
            "final_verdict": "FAKE",
            "confidence": min(0.9, refutes / max(len(analyses), 1))
        }

    if supports > refutes:
        return {
            "final_verdict": "REAL",
            "confidence": min(0.9, supports / max(len(analyses), 1))
        }

    return {
        "final_verdict": "UNKNOWN",
        "confidence": 0.5
    }


In [33]:
user_claim = "Drinking bleach cures COVID-19"

news = fetch_news_with_features(user_claim, num_results=5)

analyses = [
    analyze_news_llama3(item, user_claim)
    for item in news
]

final = aggregate_claim_verdict(analyses)

print(final)


{'final_verdict': 'FAKE', 'confidence': 0.8}


In [34]:
user_claim = "US forces kidnaps venezuela president"

news = fetch_news_with_features(user_claim, num_results=5)

analyses = [
    analyze_news_llama3(item, user_claim)
    for item in news
]

final = aggregate_claim_verdict(analyses)

print(final)


{'final_verdict': 'REAL', 'confidence': 0.7}


In [36]:
user_claim = "Rahul Gandhi became 23 rd prime minister of India"

news = fetch_news_with_features(user_claim, num_results=5)

analyses = [
    analyze_news_llama3(item, user_claim)
    for item in news
]

final = aggregate_claim_verdict(analyses)

print(final)


{'final_verdict': 'FAKE', 'confidence': 0.3}


In [37]:
analyses

[{'relationship': 'NOT_ADDRESSED',
  'verdict': 'FAKE',
  'confidence': 0.95,
  'reasoning': 'The article does not mention Rahul Gandhi becoming the prime minister of India, but rather discusses questions about his leadership within the Congress party and among its allies. Since Rahul Gandhi has never been elected as the Prime Minister of India, the claim is false.'},
 {'relationship': 'NOT_ADDRESSED',
  'verdict': 'FAKE',
  'confidence': 1,
  'reasoning': "The article does not mention Rahul Gandhi or his becoming the prime minister. It talks about Narendra Modi, indicating that he is currently the Prime Minister of India, which contradicts the user's claim."},
 {'relationship': 'NOT_ADDRESSED',
  'verdict': 'FAKE',
  'confidence': 0.95,
  'reasoning': 'The article does not mention Rahul Gandhi becoming the prime minister of India. It discusses his accusation against BJP regarding election fraud in Bihar, which is unrelated to him being the 23rd prime minister.'},
 {'relationship': 'RE